
## **Week 5 Wednesday Data Validation: Type Checking, Sanitization, Error Handling — Comparing XML vs JSON Parsing**


**What you'll practice today**

- Validate data types & required fields (JSON & XML)

- Sanitize/clean inputs (strings, HTML-like text)

- Handle errors safely (graceful failures, informative messages)

- Compare **JSON** parsing with **XML (ElementTree)** parsing

- Prepare data for your Core Project (API → DB → CRUD)


In [ ]:
##### 월요일 / 수요일 수업 내용 복습!!! -> 20일(마감일) 과제 2/11 당장 시작 -> 2/12 질문 포함 (교수님)께 이메일 보내기 
# 교수님께 보낼 내용
# - 레쥬메 봐주실 수 있는지 
# - 


### **Warm‑up: Why validate & sanitize?** 

- External inputs (APIs, user forms, files) are **often wrong or malicious**.

- Your program should **check types**, **clean up text**, and **handle errors** rather than crash.

- Validation happens **at the boundary**: when you receive the data.



---
### **1) JSON parsing + basic validation** 

We'll use Python's built‑in `json` module:
- `json.loads(str)` → parse a JSON **string** into Python types  

- `json.dumps(obj)` → turn Python object into a JSON string

We'll practice:

- Checking **required keys**

- Checking **value types** (int/str/list)

- Failing safely with **`try/except json.JSONDecodeError`**


In [ ]:

import json

good_json = '{"userId": 1, "title": "Hello", "body": "World"}'
bad_json  = '{"userId": 1, "title": "Hello", "body": "World"  '  # missing closing brace
##### (prof) 과제할 때 다 사용 가능함 
def parse_json(s: str):
    """Return dict on success, or {} on parse failure (and print why)."""
    try:
        return json.loads(s)
    except json.JSONDecodeError as e: #### 교수님 말하시는 바가 이걸 항상 표시하라고 하는 것 같음 
        # there are so many requests, so you have to use this!!!
        # THESE ARE GOOD THING, SO I HAVE TO USE ALL OF THESE!!!
        print("JSON parse error:", e) ##### (prof) do whatever by this 
        return {}

print(parse_json(good_json))
print(parse_json(bad_json))  # triggers JSONDecodeError


# Saved ids - 200K
# id ---(api)---> result 

{'userId': 1, 'title': 'Hello', 'body': 'World'}
JSON parse error: Expecting ',' delimiter: line 1 column 50 (char 49)
{}



### Validate expected structure & types
Rule: expect keys `userId:int`, `title:str (nonempty)`, `body:str`.


In [ ]:

def validate_post(data: dict) -> tuple[bool, str]: ##### not composite 
    # Check required keys and types
    if not isinstance(data, dict):
        return False, "Not a JSON object"
    if "userId" not in data or not isinstance(data["userId"], int): ##### you have to check user's id expecting it's 'int'
        #### YOU HAVE TO CKECK ALL TYPES, if the instance type is int?? 
        return False, "userId must be an integer"
    if "title" not in data or not isinstance(data["title"], str) or not data["title"].strip():
        return False, "title must be a nonempty string"
    if "body" not in data or not isinstance(data["body"], str):
        return False, "body must be a string" 
    return True, "OK" ##### exceptually ##### you also print like this, return value message 

print(validate_post(parse_json('{"userId":2,"title":" T ","body":"text"}'))) ##### this one looks like @@@
print(validate_post(parse_json('{"userId":"2","title":"x","body":"y"}')))  # wrong type

##### just mee up these requirements 

(True, 'OK')
(False, 'userId must be an integer')


- In larger apps, you can use **JSON Schema** or libraries like **Pydantic** to automate type checks. 

- Here we focus on core Python so everyone can run it.



---
### **2) Sanitization**

**Sanitization** = making input **safe/clean** before storing or rendering.

Examples:

- Trim whitespace; collapse repeated spaces

- Limit length (avoid huge strings)

- Remove or **escape** HTML/JS if you will render it

- Allow only a safe character set (whitelisting) when appropriate


In [ ]:

import re
import html
##### (prof) html skip, you skip, all these anything 
# <body> </body>
# <head? </head>
# 100 requests in a m@@

def sanitize_title(s: str, max_len: int = 80) -> str:
    # Trim & collapse spaces
    s = " ".join(s.split())
    # Limit length
    s = s[:max_len]
    return s

def escape_html(s: str) -> str: # 이렇게 사용하기 위해서는 위처럼 html package를 사용해야 한다 
    # Escape <, >, &, quotes so it's safe to render
    return html.escape(s, quote=True)

def whitelist_basic_text(s: str) -> str:
    # Keep letters, numbers, spaces, common punctuation
    return re.sub(r"[^\w\s.,;:!?\-()'\"]+", "", s) 

raw = '  <script>alert("XSS")</script>   Hello   World!!!   '  ##### we don't want to @@@, when api call, we don't use like above this 

print("escape_html:", escape_html(raw))
print("sanitize_title:", sanitize_title(raw))
print("whitelist_basic_text:", whitelist_basic_text(raw))


escape_html:   &lt;script&gt;alert(&quot;XSS&quot;)&lt;/script&gt;   Hello   World!!!   
sanitize_title: <script>alert("XSS")</script> Hello World!!!
whitelist_basic_text:   scriptalert("XSS")script   Hello   World!!!   



If you truly must accept/keep some HTML (e.g., blog content), use a well‑tested sanitizer (e.g., **Bleach**) with an 

**allow‑list** of tags/attrs. Keep it simple for first‑year work.



---
### **3) Error handling patterns**

Use `try/except` where things can fail (network, parsing, type conversion). Return helpful messages or defaults instead of crashing.


In [4]:

def safe_int(x, default=None):
    try:
        return int(x)
    except (TypeError, ValueError):
        return default

def load_post(json_str: str) -> dict:
    data = parse_json(json_str)
    ok, msg = validate_post(data)
    if not ok:
        return {"error": msg}
    # sanitize fields that may be displayed/stored
    data["title"] = sanitize_title(data["title"])
    data["body"]  = whitelist_basic_text(data["body"])
    return data

print(load_post('{"userId":3,"title":"   Hello    CSIS   ","body":"Welcome!!! <b>bold</b>"}'))
print(load_post('{"userId":"oops","title":"x","body":"y"}'))  # type error path


{'userId': 3, 'title': 'Hello CSIS', 'body': 'Welcome!!! bboldb'}
{'error': 'userId must be an integer'}



---
### **4) XML parsing & validation with ElementTree**

**ElementTree** is in the standard library and reads XML into a tree of elements:

- Parse from **string**: `ET.fromstring(xml_str)`  

- Parse from **file**: `ET.parse(path).getroot()` 

- Access: `elem.tag`, `elem.attrib`, `elem.text`  

- Find: `elem.find('child')`, `elem.findall('child')`

We'll parse a small `<library>` and validate books.


In [5]:

import xml.etree.ElementTree as ET

xml_good = '''<?xml version="1.0"?>
<library>
  <book id="1" genre="fiction"><title>1984</title><pages>328</pages></book>
  <book id="2" genre="nonfiction"><title>Sapiens</title><pages>443</pages></book>
</library>'''

xml_bad  = '''<library>
  <book id="X" genre="fiction"><title></title><pages>abc</pages></book>
</library>'''

def parse_xml(s: str):
    try:
        return ET.fromstring(s)
    except ET.ParseError as e:
        print("XML parse error:", e)
        return None

def validate_book_elem(elem: ET.Element) -> tuple[bool, str]:
    # expected: attributes id:int, genre:str; children title:str(nonempty), pages:int
    bid = elem.get("id")
    genre = elem.get("genre")
    title_el = elem.find("title")
    pages_el = elem.find("pages")
    if bid is None or genre is None or title_el is None or pages_el is None:
        return False, "Missing attribute or child element"
    # types
    try:
        int(bid)
    except ValueError:
        return False, "id must be an integer"
    if not title_el.text or not title_el.text.strip():
        return False, "title must be nonempty"
    try:
        int((pages_el.text or "").strip())
    except ValueError:
        return False, "pages must be an integer"
    return True, "OK"

root_ok = parse_xml(xml_good)
if root_ok is not None:
    for b in root_ok.findall("book"):
        print("GOOD:", validate_book_elem(b))

root_bad = parse_xml(xml_bad)
if root_bad is not None:
    for b in root_bad.findall("book"):
        print("BAD:", validate_book_elem(b))


GOOD: (True, 'OK')
GOOD: (True, 'OK')
BAD: (False, 'id must be an integer')



### JSON vs XML — practical comparison

- **JSON**: compact, direct Python mapping; native numbers/booleans; easier for many web APIs.

- **XML**: verbose, attributes + elements; **all text** so you must cast to numbers/booleans; powerful schemas (DTD/XSD) used in some systems.

- Safety: disallow or avoid external entities/features in untrusted XML; for this class use **`ElementTree`** with simple strings and don’t load external references.



---
### **5) Conversions (JSON ↔ Python, XML → dict → JSON)** 

Converting to **dict/JSON** makes it easier to store in a database or send to your own API.


In [6]:

# JSON → Python → JSON
obj = parse_json('{"a":1,"b":[1,2,3]}')
s = json.dumps(obj, indent=2, sort_keys=True)
print(s)

# XML → dict list → JSON
def books_to_list(root: ET.Element):
    out = []
    for b in root.findall("book"):
        title = (b.findtext("title") or "").strip()
        pages = (b.findtext("pages") or "").strip()
        try:
            pages = int(pages)
        except ValueError:
            pages = None
        out.append({
            "id": b.get("id"),
            "genre": b.get("genre"),
            "title": title,
            "pages": pages
        })
    return out

if root_ok is not None:
    as_list = books_to_list(root_ok)
    print(json.dumps(as_list, indent=2))


{
  "a": 1,
  "b": [
    1,
    2,
    3
  ]
}
[
  {
    "id": "1",
    "genre": "fiction",
    "title": "1984",
    "pages": 328
  },
  {
    "id": "2",
    "genre": "nonfiction",
    "title": "Sapiens",
    "pages": 443
  }
]



---
### **6) Quick Quiz**

1. Which exception does `json.loads` raise on invalid JSON?  

2. Name one reason **XML** validation can be trickier than **JSON** validation.  

3. Give one example of **sanitization** you’d apply before rendering text.  

4. Where should you validate inputs in an application (core vs boundary)? Why? 
 
5. True/False: In XML, numbers arrive as numbers (not strings).



---
### **7) Best Practices**
- Validate early, reject bad data fast.  

- Log errors for developers, but show friendly messages to users. 

- Sanitize text that will be rendered or stored.

- Convert types explicitly (e.g., `int(...)`) and handle failures.  

- Prefer simple, well‑documented formats; convert to JSON before DB.



---
### **8) Class Exercise**

**A. JSON task**  
Write `clean_post(json_str)` that:
1) Parses JSON safely.  
2) Validates `userId:int`, `title:str nonempty`, `body:str`.  
3) Sanitizes `title` and `body` (trim/collapse spaces; remove unusual characters).  
4) Returns the cleaned dict **or** `{"error": "...message..."}`.

**B. XML task**  
Write `collect_valid_books(xml_str)` that:
1) Parses XML safely.  
2) Validates each `<book>` using `validate_book_elem`.  
3) Returns a **list of dicts** with only valid books (id:int, genre, title, pages:int).

> Place your solutions in the cell below.


In [ ]:

# 🧑‍💻 Your turn


# **A. JSON task**  
# Write `clean_post(json_str)` that:
# 1) Parses JSON safely.  
# 2) Validates `userId:int`, `title:str nonempty`, `body:str`.  
# 3) Sanitizes `title` and `body` (trim/collapse spaces; remove unusual characters).  
# 4) Returns the cleaned dict **or** `{"error": "...message..."}`.


# 
# ite (== separate??)
def validate_id(data: dict) -> tuple[bool, str]: 
    # key
    if not isinstance(data, dict):
        return False, "Not a JSON object"
    if "userId" not in data or not isinstance(data["userId"], int): 
        return False, "userId must be an integer"
    return True, "OK"

def validate_title(data: dict) -> tuple[bool, str]: 
    # key
    if not isinstance(data, dict):
        return False, "Not a JSON object"
    if "title" not in data or not isinstance(data["title"], str) or not data["title"].strip():
        return False, "title must be a nonempty string"
    return True, "OK"

def validate_body(data: dict) -> tuple[bool, str]: 
    # key
    if not isinstance(data, dict):
        return False, "Not a JSON object"
    if "body" not in data or not isinstance(data["body"], str):
        return False, "body must be a string" 
    return True, "OK"

def sanitize_title(s: str, max_len: int = 80) -> str:
    s = " ".join(s.split())
    
    
    
    s = s[:max_len]
    return s

def clean_post(json_str: str) -> dict:
    # TODO: implement using parse_json, validate_post, and sanitizers
    return json.loads(json_str)



# **B. XML task**  
# Write `collect_valid_books(xml_str)` that:
# 1) Parses XML safely.  
# 2) Validates each `<book>` using `validate_book_elem`.  
# 3) Returns a **list of dicts** with only valid books (id:int, genre, title, pages:int).

# > Place your solutions in the cell below.

def collect_valid_books(xml_str: str):
    # TODO: parse with parse_xml, loop books, reuse validate_book_elem, build list
    pass

# Optional quick tests:
# print(clean_post('{"userId":4,"title":"  Hi   ","body":"<b>ok</b>"}'))
# print(collect_valid_books('<?xml version="1.0"?><library><book id="9" genre="f"><title>X</title><pages>10</pages></book></library>'))
